# 6. EDA multivariado

Se revisan correlaciones, asociación entre categóricas, VIF, PCA y outliers
multivariados mediante Isolation Forest. Todos los ajustes exploratorios usan
únicamente training y no provocan eliminación automática de observaciones.

In [ ]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

train = load_train()
frame = prepare_frame(train)
num_cols, cat_cols = column_types(frame)
y = frame[TARGET]

## Correlación entre variables numéricas

In [ ]:
X = frame[num_cols]
pear, spear = X.corr(method="pearson"), X.corr(method="spearman")
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, m, nombre in zip(axes, [pear, spear], ["Pearson", "Spearman"]):
    sns.heatmap(m, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax, cbar=False, annot_kws={"size": 8})
    ax.set_title(f"Correlación de {nombre}")
plt.tight_layout(); save_figure(fig, "06_correlaciones_numericas.png"); plt.show()

pares = [(a, b, pear.loc[a, b], spear.loc[a, b]) for i, a in enumerate(num_cols) for b in num_cols[i + 1:]]
pares = pd.DataFrame(pares, columns=["variable A", "variable B", "Pearson", "Spearman"])
pares["|r| máx"] = pares[["Pearson", "Spearman"]].abs().max(axis=1)
pares = pares.sort_values("|r| máx", ascending=False).reset_index(drop=True)
pares.to_csv(TABLES / "correlaciones_pares.csv", index=False)
display(pares.head(6).round(3))
altos = pares[pares["|r| máx"] > 0.7]
md_out(f"Pares con |r| > 0,7: **{len(altos)}**. " + ("Ninguno: las variables numéricas aportan información distinta y no hay multicolinealidad fuerte entre ellas."
       if len(altos) == 0 else "Conviene vigilarlos: " + ", ".join(f"`{r['variable A']}`–`{r['variable B']}`" for _, r in altos.iterrows()) + ".") +
       f" La correlación más alta es {es(pares.loc[0, '|r| máx'], 2)} entre `{pares.loc[0, 'variable A']}` y `{pares.loc[0, 'variable B']}`.")

## Asociación entre variables categóricas

V de Cramér entre pares de categóricas de baja cardinalidad (hasta 40 niveles, sin las constantes). Un valor alto indica variables casi
redundantes.

In [ ]:
sel = [c for c in cat_cols if c not in ("diag_1", "diag_2", "diag_3", "weight") and 2 <= frame[c].nunique(dropna=True) <= 40
       and frame[c].value_counts(normalize=True, dropna=False).iloc[0] < 0.99][:26]
mat = pd.DataFrame(np.eye(len(sel)), index=sel, columns=sel)
for i, a in enumerate(sel):
    for b in sel[i + 1:]:
        v = cramers_v(frame[a], frame[b])[0]
        mat.loc[a, b] = mat.loc[b, a] = v
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(mat, cmap="viridis", vmin=0, vmax=1, ax=ax, xticklabels=True, yticklabels=True, cbar_kws={"label": "V de Cramér"})
ax.set_title("Asociación entre variables categóricas"); plt.xticks(fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout(); save_figure(fig, "06_cramer_categoricas.png"); plt.show()

pares_c = [(a, b, mat.loc[a, b]) for i, a in enumerate(sel) for b in sel[i + 1:]]
pares_c = pd.DataFrame(pares_c, columns=["variable A", "variable B", "V de Cramér"]).sort_values("V de Cramér", ascending=False).reset_index(drop=True)
pares_c.to_csv(TABLES / "cramer_pares.csv", index=False)
display(pares_c.head(8).round(3))
fuertes = pares_c[pares_c["V de Cramér"] >= 0.5]
md_out(f"Pares con V ≥ 0,5: **{len(fuertes)}**. " + ("Ninguno." if len(fuertes) == 0 else
       "Son candidatos a redundancia: " + ", ".join(f"`{r['variable A']}`–`{r['variable B']}`" for _, r in fuertes.head(6).iterrows()) + ".") +
       " Los diagnósticos agrupados y los medicamentos combinados son los sospechosos habituales de redundancia.")

## Multicolinealidad: factor de inflación de la varianza (VIF)

El VIF de una variable es 1 / (1 − R²) al regresarla contra las demás numéricas. Se usan las variables estandarizadas (con `log1p` en los
conteos con cola larga). Valores superiores a 5 (o a 10) indican colinealidad preocupante.

In [ ]:
from sklearn.preprocessing import StandardScaler
logged = [c for c in ZERO_INFLATED_COUNTS if c in num_cols]
Xv = frame[num_cols].copy()
Xv[logged] = np.log1p(Xv[logged])
Z = StandardScaler().fit_transform(Xv.fillna(Xv.median()))
vif = pd.Series(variance_inflation_factors(Z), index=num_cols, name="VIF").sort_values(ascending=False)
vif.to_csv(TABLES / "vif_numericas.csv")
display(vif.round(2).to_frame())
md_out(f"El VIF máximo es **{es(vif.max(), 2)}** (`{vif.idxmax()}`). " + ("Todos están por debajo de 5: no hay colinealidad problemática entre las numéricas." if vif.max() < 5
       else "Hay variables por encima de 5: se vigilarán los coeficientes de la regresión logística."))

## Componentes principales (PCA)

Solo con las variables numéricas (con `log1p` en los conteos y estandarizadas). Sirve para ver cuánta información comparten y si las clases
se separan en las primeras direcciones.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(random_state=RANDOM_STATE).fit(Z)
var = pd.DataFrame({"componente": [f"PC{i + 1}" for i in range(len(num_cols))], "varianza explicada %": 100 * pca.explained_variance_ratio_,
                    "acumulada %": 100 * np.cumsum(pca.explained_variance_ratio_)})
var.to_csv(TABLES / "pca_varianza.csv", index=False)
display(var.round(2))
comp = pca.transform(Z)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, len(var) + 1), var["acumulada %"], "o-")
axes[0].axhline(90, color="gray", ls="--"); axes[0].set_xlabel("componente"); axes[0].set_ylabel("% de varianza acumulada")
axes[0].set_title("Varianza explicada acumulada")
axes[1].scatter(comp[:, 0], comp[:, 1], c=np.where(y.to_numpy() == 1, "#e45756", "#4c78a8"), s=2, alpha=0.12)
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2"); axes[1].set_title("Training completo (rojo = reingreso <30)")
plt.tight_layout(); save_figure(fig, "06_pca.png"); plt.show()
carga = pd.DataFrame(pca.components_[:2].T, index=num_cols, columns=["PC1", "PC2"]).round(2)
n90 = int((var['acumulada %'] < 90).sum()) + 1
auc_pc = max(univariate_auc(pd.Series(comp[:, i]), y) for i in range(2))
md_out(f"Se necesitan **{n90} de {len(num_cols)} componentes** para explicar 90 % de la varianza numérica. El mejor AUC aislado de PC1/PC2 es {es(auc_pc, 3)}; PCA maximiza varianza sin usar el target, por lo que el solapamiento no demuestra que el problema sea inseparable.")
carga

## Outliers multivariados: Isolation Forest

Isolation Forest se ajusta con todas las filas de training y las variables
numéricas imputadas, transformadas y estandarizadas del análisis anterior.
`contamination='auto'` evita imponer una tasa arbitraria. La etiqueta es un
diagnóstico algorítmico, no evidencia de error clínico.

In [ ]:
from sklearn.ensemble import IsolationForest

isolation = IsolationForest(n_estimators=200, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1)
iso_label = isolation.fit_predict(Z)
iso_score = isolation.decision_function(Z)
flagged = iso_label == -1
outlier_summary = pd.DataFrame({
    "grupo": ["training completo", "clase 0", "reingreso <30"],
    "n": [len(frame), int((y == 0).sum()), int((y == 1).sum())],
    "marcados": [int(flagged.sum()), int(flagged[y.to_numpy() == 0].sum()), int(flagged[y.to_numpy() == 1].sum())],
    "proporción_marcada": [flagged.mean(), flagged[y.to_numpy() == 0].mean(), flagged[y.to_numpy() == 1].mean()],
})
outlier_summary.to_csv(TABLES / "isolation_forest_resumen.csv", index=False)
display(outlier_summary.round(4))

med_regular = frame.loc[~flagged, num_cols].median()
med_anomaly = frame.loc[flagged, num_cols].median()
scale = frame[num_cols].std().replace(0, np.nan)
profile = ((med_anomaly - med_regular) / scale).abs().sort_values(ascending=False).rename("diferencia_mediana_en_DE").to_frame()
profile.to_csv(TABLES / "isolation_forest_perfil.csv")
display(profile.head(8).round(3))
md_out(f"Isolation Forest marca **{es(flagged.mean(), 2, pct=True)}** del training. La proporción es {es(flagged[y.to_numpy() == 1].mean(), 2, pct=True)} en `<30` y {es(flagged[y.to_numpy() == 0].mean(), 2, pct=True)} en clase 0. Se conservan todas las filas: los extremos pueden representar pacientes reales de alta complejidad.")

## ¿Clustering?

No se fuerza clustering en este entregable. La pregunta es supervisada, las
variables mezclan códigos administrativos y medidas clínicas, y no existe
una hipótesis de segmentos clínicos validable con la información disponible.
PCA e Isolation Forest cubren el objetivo exploratorio multivariado requerido.